In [1]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_forestdiffusion


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import evaluate_quality



# ----------------------------------------------------
# Load Dataset
# ----------------------------------------------------

magic_gamma_telescope = fetch_ucirepo(id=159)

# data (as pandas dataframes)
X = magic_gamma_telescope.data.features
y = magic_gamma_telescope.data.targets

# Combine features and target into a single DataFrame
data = pd.concat([X, y], axis=1)

target_col = y.columns[0] # Correctly identify the target column

# metadata
print("Dataset Metadata:")
print(magic_gamma_telescope.metadata)

# variable information
print("\nDataset Variable Information:")
print(magic_gamma_telescope.variables)

# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42
RUN_QUALITY_EVAL = True

# Diffusion generators only
GENERATORS_TO_EVAL = ["ForestDiffusion", "TabDDPM"]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------------------------------
# Storage Containers
# ----------------------------------------------------
scores = {}

synthetic_datasets = {}

quality_results = []


Dataset Metadata:
{'uci_id': 159, 'name': 'MAGIC Gamma Telescope', 'repository_url': 'https://archive.ics.uci.edu/dataset/159/magic+gamma+telescope', 'data_url': 'https://archive.ics.uci.edu/static/public/159/data.csv', 'abstract': 'Data are MC generated to simulate registration of high energy gamma particles in an atmospheric Cherenkov telescope', 'area': 'Physics and Chemistry', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 19020, 'num_features': 10, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2004, 'last_updated': 'Tue Dec 19 2023', 'dataset_doi': '10.24432/C52C8B', 'creators': ['R. Bock'], 'intro_paper': None, 'additional_info': {'summary': "The data are MC generated (see below) to simulate registration of high energy gamma particles in a ground-based atmospheric Cherenkov gamma telescope using the imaging techniq

In [4]:
# SINGLE RUN

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


# TRAIN / TEST SPLIT (NO LEAKAGE)

train_real, test_real = train_test_split(
    data,
    test_size=TEST_SIZE,
    stratify=data[target_col],
    random_state=seed
)

train_diffusion_metadata = SingleTableMetadata()
train_diffusion_metadata.detect_from_dataframe(train_real)

# ---------------------------------------------------
# TabDDPM
# ---------------------------------------------------

if 'TabDDPM' in GENERATORS_TO_EVAL:
    try:

        print("Training TabDDPM...")
        synthetic_tabddpm = train_tabddpm(
            train_real,
            target_col=target_col,
            categorical_columns=[target_col],
            n_samples=N_SAMPLES,
            seed=seed,
        )

        synthetic_datasets["TabDDPM"] = synthetic_tabddpm.copy()

        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_tabddpm,
                metadata=train_diffusion_metadata,
            )

            scores["TabDDPM"] = quality.get_score()

            print("TabDDPM:", round(scores["TabDDPM"], 4))
        else:
            print("TabDDPM: trained (quality eval skipped)")

    except Exception as e:
        print("TabDDPM Failed:", e)
else:
    print("TabDDPM: skipped (not in GENERATORS_TO_EVAL)")



================ SINGLE RUN ================
Training TabDDPM...
[0]
12
{'num_classes': 2, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 256, 256], 'dropout': 0.0}, 'd_in': np.int64(12)}
mlp
Step 500/1000 MLoss: 0.0 GLoss: 0.3253 Sum: 0.3253
Step 1000/1000 MLoss: 0.0 GLoss: 0.312 Sum: 0.312
mlp
Sample timestep    0
Discrete cols: []
Num shape:  (1000, 10)
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 382.19it/s]|
Column Shapes Score: 93.74%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 202.77it/s]|
Column Pair Trends Score: 93.31%

Overall Score (Average): 93.52%

TabDDPM: 0.9352


In [5]:
# ForestDiffusion

if 'ForestDiffusion' in GENERATORS_TO_EVAL:
    try:
        import traceback

        print("Training ForestDiffusion...")
        synthetic_forestdiffusion = train_forestdiffusion(
            train_real,
            target_col=target_col,
            categorical_columns=[target_col],
            n_samples=N_SAMPLES,
            seed=seed,
        )

        synthetic_datasets["ForestDiffusion"] = synthetic_forestdiffusion.copy()

        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_forestdiffusion,
                metadata=train_diffusion_metadata,
            )

            scores["ForestDiffusion"] = quality.get_score()

            print("ForestDiffusion:", round(scores["ForestDiffusion"], 4))
        else:
            print("ForestDiffusion: trained (quality eval skipped)")

        del synthetic_forestdiffusion

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:

        print("ForestDiffusion Failed:")
        traceback.print_exc()
else:
    print("ForestDiffusion: skipped (not in GENERATORS_TO_EVAL)")


Training ForestDiffusion...
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 371.92it/s]|
Column Shapes Score: 96.83%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 212.28it/s]|
Column Pair Trends Score: 96.34%

Overall Score (Average): 96.58%

ForestDiffusion: 0.9658


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': LinearSVC(max_iter=2000, dual='auto', random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=[42,43,44,45,46,47,48,49,50,51]
):

    results = []

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_train, _, y_train, _ = train_test_split(
                X_train,
                y_train,
                test_size=test_size,
                random_state=seed,
                stratify=y_train
            )

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            _, X_test, _, y_test = train_test_split(
                X_test,
                y_test,
                test_size=test_size,
                random_state=seed,
                stratify=y_test
            )

            scaler = StandardScaler().fit(X_train)

            X_train_s = scaler.transform(X_train)
            X_test_s = scaler.transform(X_test)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            clf.fit(X_train_s, y_train)

            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(
                accuracy_score(y_test, y_pred)
            )

            f1_scores.append(
                f1_score(
                    y_test,
                    y_pred,
                    pos_label="g",
                    average="binary",
                    zero_division=0
                )
            )

            precision_scores.append(
                precision_score(
                    y_test,
                    y_pred,
                    pos_label="g",
                    average="binary",
                    zero_division=0
                )
            )

            recall_scores.append(
                recall_score(
                    y_test,
                    y_pred,
                    pos_label="g",
                    average="binary",
                    zero_division=0
                )
            )

        results.append({
            "Model": name,

            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": np.std(accuracy_scores),

            "F1 Mean": np.mean(f1_scores),
            "F1 Std": np.std(f1_scores),

            "Precision Mean": np.mean(precision_scores),
            "Precision Std": np.std(precision_scores),

            "Recall Mean": np.mean(recall_scores),
            "Recall Std": np.std(recall_scores),

            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} \u00b1 {np.std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} \u00b1 {np.std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} \u00b1 {np.std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} \u00b1 {np.std(recall_scores):.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False
    )


In [8]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd

label_col = target_col

model_order = [
    "ForestDiffusion",
    "TabDDPM",
]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

print("TRTR (Train Real, Test Real)")
print(f"Diffusion generators: {model_order}")

trtr_results = evaluate_models(
    train_df=data,
    test_df=data,
    label_col="class",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"Skipping {synth_name} — not in synthetic_datasets")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=data,
        label_col="class",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=["_TRTR", "_TSTR"]
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)
Diffusion generators: ['ForestDiffusion', 'TabDDPM']


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
5,RandomForest,0.8803 ± 0.0056,0.9106 ± 0.0042,0.8825 ± 0.0041,0.9407 ± 0.0051
6,ExtraTrees,0.8783 ± 0.0062,0.9100 ± 0.0046,0.8740 ± 0.0054,0.9492 ± 0.0053
9,MLP,0.8780 ± 0.0050,0.9089 ± 0.0038,0.8810 ± 0.0056,0.9386 ± 0.0075
7,GradientBoost,0.8715 ± 0.0048,0.9054 ± 0.0034,0.8657 ± 0.0043,0.9490 ± 0.0035
2,KNN,0.8379 ± 0.0045,0.8826 ± 0.0032,0.8318 ± 0.0037,0.9401 ± 0.0043
8,AdaBoost,0.8261 ± 0.0073,0.8699 ± 0.0062,0.8442 ± 0.0063,0.8974 ± 0.0139
4,DecisionTree,0.8164 ± 0.0046,0.8581 ± 0.0034,0.8599 ± 0.0051,0.8564 ± 0.0040
0,LogReg,0.7909 ± 0.0075,0.8477 ± 0.0054,0.8032 ± 0.0059,0.8974 ± 0.0058
1,SVM-RBF,0.7892 ± 0.0076,0.8469 ± 0.0054,0.8004 ± 0.0060,0.8991 ± 0.0057
3,NaiveBayes,0.7257 ± 0.0080,0.8125 ± 0.0053,0.7297 ± 0.0055,0.9165 ± 0.0061


ForestDiffusion - TSTR


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
5,RandomForest,0.8540 ± 0.0072,0.8910 ± 0.0050,0.8633 ± 0.0086,0.9207 ± 0.0048
6,ExtraTrees,0.8531 ± 0.0049,0.8919 ± 0.0035,0.8529 ± 0.0051,0.9345 ± 0.0054
7,GradientBoost,0.8492 ± 0.0082,0.8867 ± 0.0061,0.8643 ± 0.0078,0.9103 ± 0.0079
9,MLP,0.8298 ± 0.0071,0.8682 ± 0.0059,0.8715 ± 0.0062,0.8651 ± 0.0110
8,AdaBoost,0.8167 ± 0.0061,0.8608 ± 0.0054,0.8476 ± 0.0065,0.8746 ± 0.0131
2,KNN,0.8024 ± 0.0063,0.8591 ± 0.0045,0.7990 ± 0.0049,0.9289 ± 0.0065
0,LogReg,0.7915 ± 0.0060,0.8475 ± 0.0043,0.8062 ± 0.0054,0.8932 ± 0.0059
1,SVM-RBF,0.7897 ± 0.0054,0.8466 ± 0.0038,0.8032 ± 0.0053,0.8949 ± 0.0062
4,DecisionTree,0.7876 ± 0.0116,0.8348 ± 0.0102,0.8417 ± 0.0095,0.8283 ± 0.0186
3,NaiveBayes,0.7210 ± 0.0078,0.8064 ± 0.0057,0.7329 ± 0.0048,0.8963 ± 0.0087


ForestDiffusion - TRTR vs TSTR


...

,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
9,MLP,0.6626 ± 0.0169,0.7683 ± 0.0127,0.6922 ± 0.0097,0.8633 ± 0.0200
0,LogReg,0.6549 ± 0.0045,0.7897 ± 0.0020,0.6527 ± 0.0031,0.9993 ± 0.0009
1,SVM-RBF,0.6547 ± 0.0048,0.7896 ± 0.0022,0.6527 ± 0.0034,0.9992 ± 0.0012
8,AdaBoost,0.6441 ± 0.0075,0.7743 ± 0.0071,0.6574 ± 0.0043,0.9420 ± 0.0224
5,RandomForest,0.6350 ± 0.0110,0.7603 ± 0.0080,0.6619 ± 0.0059,0.8932 ± 0.0139
6,ExtraTrees,0.6347 ± 0.0133,0.7597 ± 0.0090,0.6622 ± 0.0077,0.8911 ± 0.0128
7,GradientBoost,0.6291 ± 0.0097,0.7533 ± 0.0076,0.6621 ± 0.0056,0.8738 ± 0.0160
2,KNN,0.6023 ± 0.0122,0.7154 ± 0.0098,0.6673 ± 0.0076,0.7711 ± 0.0146
4,DecisionTree,0.5681 ± 0.0095,0.6630 ± 0.0078,0.6709 ± 0.0093,0.6554 ± 0.0134
3,NaiveBayes,0.4102 ± 0.1251,0.3678 ± 0.2273,0.5062 ± 0.1185,0.3476 ± 0.3317


TabDDPM - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy (Mean±Std)_TRTR,Accuracy (Mean±Std)_TSTR
0,TabDDPM,RandomForest,0.245321,0.150323,0.220576,0.047486,0.8803 ± 0.0056,0.6350 ± 0.0110
1,TabDDPM,ExtraTrees,0.243665,0.150294,0.211826,0.058110,0.8783 ± 0.0062,0.6347 ± 0.0133
2,TabDDPM,MLP,0.215431,0.140580,0.188806,0.075304,0.8780 ± 0.0050,0.6626 ± 0.0169
3,TabDDPM,GradientBoost,0.242429,0.152154,0.203586,0.075264,0.8715 ± 0.0048,0.6291 ± 0.0097
4,TabDDPM,KNN,0.235568,0.167238,0.164527,0.169019,0.8379 ± 0.0045,0.6023 ± 0.0122
5,TabDDPM,AdaBoost,0.181940,0.095641,0.186748,-0.044566,0.8261 ± 0.0073,0.6441 ± 0.0075
6,TabDDPM,DecisionTree,0.248344,0.195160,0.188932,0.201014,0.8164 ± 0.0046,0.5681 ± 0.0095
7,TabDDPM,LogReg,0.136015,0.057999,0.150425,-0.101906,0.7909 ± 0.0075,0.6549 ± 0.0045
8,TabDDPM,SVM-RBF,0.134516,0.057325,0.147734,-0.100041,0.7892 ± 0.0076,0.6547 ± 0.0048
9,TabDDPM,NaiveBayes,0.315563,0.444668,0.223453,0.568938,0.7257 ± 0.0080,0.4102 ± 0.1251


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
0,ForestDiffusion,0.019950,0.015989,0.008966,0.023779
1,TabDDPM,0.219879,0.161138,0.188661,0.094862


In [9]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
